# Pipeline End-to-End — Node-by-Node Trace

Notebook chay toan bo pipeline EduBot, phan ro **tung node** de kiem soat:

```
1 ContextAnalyzer -> 2 IntentRouter (LLM) -> 3 SessionManager -> 4 ActionPlanner -> 5 RAG Search -> 6 Handler -> 7 Session Save
```

Ket qua pipeline duoc ghi ra `pipeline_trace.log` dang **JSON** (reset moi query). File `app.log` giu persistent log.

---
## 0. Setup — Path & Environment

In [1]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")

# Add src to path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f"API Key: {'SET' if os.getenv('GENAI_API_KEY') else 'NOT SET'}")
print(f"Python: {sys.version.split()[0]}")

Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN
API Key: SET
Python: 3.12.4


---
## 1. Init — CustomSearch + Reranker + Orchestrator

In [2]:
from src.config.config import settings
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

DATA_DIR = PROJECT_ROOT / 'data'
CHUNKS_PATH = str(DATA_DIR / 'rag_chunks_v2.json')
EMBEDDINGS_PATH = str(DATA_DIR / 'embeddings.npy')

print("=" * 60)
print("[NODE 0] Initializing Components")
print("=" * 60)

# 1a. CustomSearch
t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"  CustomSearch: {searcher.corpus_size} chunks, dim={searcher.embeddings.shape[1]} ({time.time()-t0:.2f}s)")

# 1b. Reranker
reranker = Reranker()
print(f"  Reranker: {settings.RERANKER_MODEL} (lazy load)")

# 1c. Orchestrator
orch = Orchestrator(retriever=searcher, reranker=reranker)
print(f"  Orchestrator: ready")
print(f"  LLM Model: {settings.LLM_MODEL}")
print("=" * 60)

[22:42:45] INFO    | Tokenizing 2348 docs with underthesea...


[NODE 0] Initializing Components
CustomSearch initialized: 2348 docs, vocab=9672, avgdl=137.7
  CustomSearch: 2348 chunks, dim=768 (15.79s)
  Reranker: AITeamVN/Vietnamese_Reranker (lazy load)
  Orchestrator: ready
  LLM Model: gemini-2.5-flash-lite


---
## 2. Helper — Log Viewer & Node Runner

In [3]:
TRACE_LOG = PROJECT_ROOT / 'logs' / 'pipeline_trace.log'
APP_LOG = PROJECT_ROOT / 'logs' / 'app.log'

def show_trace_log():
    """Hien thi pipeline_trace.log (JSON format)."""
    if TRACE_LOG.exists():
        content = TRACE_LOG.read_text(encoding='utf-8')
        try:
            data = json.loads(content)
            print(json.dumps(data, indent=2, ensure_ascii=False, default=str))
        except json.JSONDecodeError:
            print(content)  # fallback to raw text
    else:
        print('[!] pipeline_trace.log chua ton tai')

def load_trace_json() -> dict:
    """Load pipeline_trace.log as dict."""
    if TRACE_LOG.exists():
        return json.loads(TRACE_LOG.read_text(encoding='utf-8'))
    return {}

def show_app_log(n=30):
    """Hien thi n dong cuoi cua app.log."""
    if APP_LOG.exists():
        lines = APP_LOG.read_text(encoding='utf-8').strip().split('\n')
        for line in lines[-n:]:
            print(line)
    else:
        print('[!] app.log chua ton tai')

def show_debug_info(debug_info: dict):
    """Hien thi debug info tu orchestrator.last_debug_info."""
    if not debug_info:
        print('[!] Chua co debug info')
        return
    print(json.dumps(debug_info, indent=2, ensure_ascii=False, default=str))

print('Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)')

Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)


---
## 3. Chay tung Node rieng le

Phan nay tach pipeline ra tung buoc de debug.

### 3.1 — Node: ContextAnalyzer

In [6]:
from src.llm.context_analyzer import ContextAnalyzer

analyzer = ContextAnalyzer()

# Test cases
# Mẫu test case mới cho 3.1:
test_cases = [
    {"query": "Mạng LAN là gì?", "history": ""}, # Độc lập -> False
    {"query": "Nó khác gì với WAN?", "history": "user: Mạng LAN là gì?\nassistant: Mạng LAN là..."}, # Đại từ "Nó" -> True
    {"query": "So sánh hai loại này", "history": "user: Giải thích hub và switch..."}, # So sánh -> True
    {"query": "Thầy có thể nói rõ hơn được không?", "history": "user: Giao thức IP..."}, # Từ tình thái "có thể" -> True
    {"query": "Vâng ạ", "history": "user: Em đã hiểu chưa?"}, # Khẳng định ngắn -> True
    {"query": "Giải thích chi tiết về tầng mạng", "history": "user: Mô hình OSI..."}, # Query đầy đủ dù có history -> False
]


print("=" * 60)
print("[NODE 1] ContextAnalyzer")
print("=" * 60)
for tc in test_cases:
    needs = analyzer.needs_contextualization(tc["query"], tc["history"])
    print(f"  Query: '{tc['query'][:50]}'")
    print(f"  History: {'Yes' if tc['history'] else 'No'}")
    print(f"  Needs context: {needs}")

[NODE 1] ContextAnalyzer
  Query: 'Mạng LAN là gì?'
  History: No
  Needs context: False
  Query: 'Nó khác gì với WAN?'
  History: Yes
  Needs context: True
  Query: 'So sánh hai loại này'
  History: Yes
  Needs context: True
  Query: 'Thầy có thể nói rõ hơn được không?'
  History: Yes
  Needs context: True
  Query: 'Vâng ạ'
  History: Yes
  Needs context: False
  Query: 'Giải thích chi tiết về tầng mạng'
  History: Yes
  Needs context: False


### 3.2 — Node: IntentRouter (LLM Call)

In [8]:
from src.llm.intent_router import IntentRouter

router = IntentRouter()

test_queries = [
    "tổng hợp kiến thức lớp 10 chủ đề 1 bộ sách cánh diều",
    "Giai thich TCP/IP la gi theo sach Ket Noi Tri Thuc",
    "Tao 3 cau trac nghiem ve mang may tinh (khong co sach)",
    "Giai thich TCP/IP la gi",
    "Xin chao ban la ai",
    "Tao slide bai an toan thong tin",
    "Cau 1 dap an A",
]

print("=" * 60)
print("[NODE 2] IntentRouter (LLM Calls)")
print("=" * 60)
for q in test_queries:
    t0 = time.time()
    result = router.detect(query=q)
    elapsed = time.time() - t0
    print(f"  Query: '{q}'")
    print(f"  Intent: {result.primary_intent} | Task: {result.task_type} | Topic: {result.topic} | New: {result.is_new_topic} | Book: {result.book}")
    print(f"  Time: {elapsed:.2f}s")

[NODE 2] IntentRouter (LLM Calls)


[00:00:02] INFO    | IntentRouter: intent=generate, task_type=lesson_plan, topic=kiến thức lớp 10 chủ đề 1, is_new_topic=True, book=CD


  Query: 'tổng hợp kiến thức lớp 10 chủ đề 1 bộ sách cánh diều'
  Intent: generate | Task: lesson_plan | Topic: kiến thức lớp 10 chủ đề 1 | New: True | Book: CD
  Time: 1.40s


[00:00:03] INFO    | IntentRouter: intent=explain, task_type=None, topic=TCP/IP, is_new_topic=True, book=KNTT


  Query: 'Giai thich TCP/IP la gi theo sach Ket Noi Tri Thuc'
  Intent: explain | Task: None | Topic: TCP/IP | New: True | Book: KNTT
  Time: 1.12s


...

  Query: 'Xin chao ban la ai'
  Intent: chat | Task: None | Topic: None | New: True | Book: None
  Time: 2.87s


[00:00:11] INFO    | IntentRouter: intent=generate, task_type=slide, topic=an toan thong tin, is_new_topic=True, book=None


  Query: 'Tao slide bai an toan thong tin'
  Intent: generate | Task: slide | Topic: an toan thong tin | New: True | Book: None
  Time: 1.08s


[00:00:13] INFO    | IntentRouter: intent=interact, task_type=None, topic=None, is_new_topic=False, book=None


  Query: 'Cau 1 dap an A'
  Intent: interact | Task: None | Topic: None | New: False | Book: None
  Time: 2.19s


### 3.3 — Node: SessionManager

In [ ]:
from src.llm.intent_router import IntentResult
from src.llm.session_manager import SessionManager
from src.llm.session_store import SessionStore
from src.llm.memory import MemoryManager

memory = MemoryManager()
store = SessionStore(storage_path=str(PROJECT_ROOT / 'data' / 'sessions'))
sm = SessionManager(session_store=store, memory=memory)

# Simulate intent results
intents = [
    IntentResult(primary_intent="chat", topic=None, is_new_topic=True),
    IntentResult(primary_intent="generate", task_type="mcq", topic="Mang may tinh", is_new_topic=True, book="CD"),
    IntentResult(primary_intent="interact", task_type=None, topic="Mang may tinh", is_new_topic=False),
]

print("=" * 60)
print("[NODE 3] SessionManager")
print("=" * 60)
for i, intent in enumerate(intents):
    session = sm.resolve_session(intent)
    print(f"  Step {i+1}: intent={intent.primary_intent}, topic={intent.topic}, new={intent.is_new_topic}, book={session.book}")
    print(f"    -> Session: {session.session_id}, topic='{session.topic}', msgs={len(session.messages)}, book={session.book}")
    print()

[21:57:42] INFO    | No current session, creating new
[21:57:42] INFO    | New session created: id=caee31ab, topic='', intent=chat
[21:57:42] INFO    | Topic changed: '' -> 'Mang may tinh', creating new session
[21:57:42] INFO    | New session created: id=62b25781, topic='Mang may tinh', intent=generate


[NODE 3] SessionManager
  Step 1: intent=chat, topic=None, new=True
    -> Session: caee31ab, topic='', msgs=0

  Step 2: intent=generate, topic=Mang may tinh, new=True
    -> Session: 62b25781, topic='Mang may tinh', msgs=0

  Step 3: intent=interact, topic=Mang may tinh, new=False
    -> Session: 62b25781, topic='Mang may tinh', msgs=0



### 3.4 — Node: ActionPlanner

In [ ]:
from src.llm.action_planner import ActionPlanner, Action
from src.llm.intent_router import IntentResult
from src.llm.memory import Session, QuizSessionState, SlideSessionState, QuestionRecord

planner = ActionPlanner()

# --- 1. Tạo Dummy Sessions mô phỏng State của người dùng ---

# 1a. State trống: Vừa vào chưa làm gì
session_no_state = Session(session_id="S1", topic="Network", intent="chat")

# 1b. State Quiz: Người dùng đang giải dở 1 bài tập trắc nghiệm
session_quiz = Session(session_id="S2", topic="Network", intent="generate")
session_quiz.quiz_state = QuizSessionState()
round1 = session_quiz.quiz_state.create_round("mcq", "Mang_may_tinh")
round1.questions.append(QuestionRecord(question_id="Q1", question_type="mcq", content={}, source="quiz"))

# 1c. State Slide: Người dùng vừa gen xong slide, có đính kèm vài bài tập dạo ở cuối slide
session_slide = Session(session_id="S3", topic="Network", intent="generate")
session_slide.slide_state = SlideSessionState()
session_slide.slide_state.add_exercise(question_type="mcq", content={}, slide_idx=0, q_idx=0)


# --- 2. Bộ kịch bản Test hóc búa (Complex Logic) ---
scenarios_complex = [
    # Sinh bài mới
    (IntentResult(primary_intent="generate", task_type="mcq"), None, "Tao 3 cau MCQ"),
    (IntentResult(primary_intent="generate", task_type="slide"), None, "Tao slide"),
    
    # Check Tương tác bám vào Session_Quiz
    (IntentResult(primary_intent="interact"), session_quiz, "ôn lại câu sai lần 2"),
    (IntentResult(primary_intent="interact"), session_quiz, "Đáp án là A nhé"),
    
    # Check Tương tác với State khác (Session_Slide)
    (IntentResult(primary_intent="interact"), session_slide, "Câu 1 tôi chọn B"),
    
    # Check Hỏi đáp bám sát ngữ cảnh vs Hỏi đáp lan man
    (IntentResult(primary_intent="explain"), session_quiz, "Tại sao câu 2 lại sai?"),
    (IntentResult(primary_intent="explain"), session_no_state, "Giải thích hộ mình khái niệm mạng LAN với"),
    
    # Check Thống kê điểm
    (IntentResult(primary_intent="analyze"), session_quiz, "Tiến độ học tập của tôi"),
]


print("=" * 60)
print("[NODE 4] ActionPlanner (Complex Logic Tests)")
print("=" * 60)

for intent, session, msg in scenarios_complex:
    plan = planner.plan(intent, session, msg)
    
    session_label = "None"
    if session:
        if session.quiz_state: session_label = "QuizState"
        elif session.slide_state: session_label = "SlideState"
        
    print(f"  Message: '{msg}'")
    print(f"  Level 1 (LLM Intent): {intent.primary_intent} | Level 2 (State Detected): {session_label}")
    
    # Nếu round_id được parse ra, hiển thị để check
    round_info = f" (round_id={plan.round_id})" if plan.round_id is not None else ""
    print(f"  -> FINAL ACTION CHOSEN: {plan.action.value}{round_info}")
    print(f"     Reason: {plan.reason}\n")


[NODE 4] ActionPlanner (Complex Logic Tests)
  Message: 'Tao 3 cau MCQ'
  Level 1 (LLM Intent): generate | Level 2 (State Detected): None
  -> FINAL ACTION CHOSEN: generate_quiz
     Reason: task_type=mcq

  Message: 'Tao slide'
  Level 1 (LLM Intent): generate | Level 2 (State Detected): None
  -> FINAL ACTION CHOSEN: generate_slide
     Reason: task_type=slide

  Message: 'ôn lại câu sai lần 2'
  Level 1 (LLM Intent): interact | Level 2 (State Detected): QuizState
  -> FINAL ACTION CHOSEN: review_wrong (round_id=1)
     Reason: Review keywords detected, round_id=1

  Message: 'Đáp án là A nhé'
  Level 1 (LLM Intent): interact | Level 2 (State Detected): QuizState
  -> FINAL ACTION CHOSEN: check_answer
     Reason: Session has quiz questions, assuming answer check

  Message: 'Câu 1 tôi chọn B'
  Level 1 (LLM Intent): interact | Level 2 (State Detected): SlideState
  -> FINAL ACTION CHOSEN: answer_exercise
     Reason: Session has slide exercises

  Message: 'Tại sao câu 2 lại sai?'
 

### 3.5 — Node: RAG Search (BM25 + Semantic + RRF -> Reranker)

In [ ]:
query = "Mang may tinh la gi va co nhung loai nao"

print("=" * 60)
print("[NODE 5] RAG Search")
print("=" * 60)

# 5a. CustomSearch (BM25 + Semantic + RRF)
t0 = time.time()
search_results = searcher.search(query, top_k=settings.RETRIEVER_TOP_K)
search_time = time.time() - t0
print(f"  CustomSearch: {len(search_results)} results ({search_time:.2f}s)")

# 5b. Reranker
t1 = time.time()
reranked = reranker.rerank(query, search_results, top_n=settings.RERANKER_TOP_N)
rerank_time = time.time() - t1
print(f"  Reranker: {len(reranked)} results ({rerank_time:.2f}s)")
print(f"  Total RAG: {search_time + rerank_time:.2f}s")
print()

# Show top chunks
print("  TOP CHUNKS:")
for i, r in enumerate(reranked):
    score = r.get('rerank_score', r.get('score', 0))
    preview = r['content'][:150].replace('\n', ' ')
    print(f"  [{i+1}] score={score:.4f}")
    print(f"      {preview}")
    print()

[NODE 5] RAG Search
Loading embedding model: dangvantuan/vietnamese-document-embedding...
Model loaded on cuda
  CustomSearch: 25 results (10.03s)
Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda
  Reranker: 5 results (30.39s)
  Total RAG: 40.41s

  TOP CHUNKS:
  [1] score=0.0008
      ``` +------------+---------------------------+------------+ | idBannhac  | tenBannhac                | idNhacsi   | +------------+--------------------

  [2] score=0.0000
      **Năm nhuận** là những năm chia hết cho 400 hoặc là những năm chia hết cho 4 nhưng không chia hết cho 100. Đặc biệt, những năm chia hết cho 3 328 được

  [3] score=0.0000
      Mảnh vườn trồng cúc đại đoá có chiều rộng m mét, chiều dài n mét. Mỗi mét vuông trồng được một khóm hoa. Mỗi khóm hoa bán được a nghìn đồng. Em hãy vi

  [4] score=0.0000
      Mệnh đề là một khẳng định có tính chất hoặc đúng hoặc sai. Ví dụ “Hà Nội là Thủ đô của Việt Nam” là một mệnh đề đúng, còn “9 là số nguyên tố” là một m

  [5]

### 3.6 — Node: Handler (LLM Generation)

In [ ]:
from src.llm.handlers.chat_handler import ChatHandler
from src.llm.handlers.explain_handler import ExplainHandler
from src.llm.handlers.question.mcq_handler import MCQHandler
from src.llm.utils import format_contexts

context_text = format_contexts(reranked)  # From previous cell

print("=" * 60)
print("[NODE 6] Handler Execution")
print("=" * 60)

# --- 6a. ChatHandler ---
print("\n--- ChatHandler ---")
chat_h = ChatHandler()
t0 = time.time()
chat_resp = chat_h.handle(query="Mang may tinh la gi?", context=context_text)
print(f"  Time: {time.time()-t0:.2f}s | Length: {len(chat_resp)}")
print(f"  Preview: {chat_resp[:300]}")
print()

[NODE 6] Handler Execution

--- ChatHandler ---
  Time: 1.80s | Length: 1018
  Preview: Chào bạn! 👋 EduBot rất vui được hỗ trợ bạn học Tin học THPT.

Câu hỏi của bạn về "Mang may tinh" có vẻ hơi khác so với các chủ đề Tin học THPT mà EduBot được học. 🤔

Trong chương trình Tin học THPT, chúng ta thường tìm hiểu về các khái niệm như:

*   **Mệnh đề và giá trị chân lý:** Như trong Context



In [ ]:
# --- 6b. ExplainHandler ---
print("--- ExplainHandler ---")
explain_h = ExplainHandler()
t0 = time.time()
explain_resp = explain_h.handle(query="Giai thich mang LAN khac gi mang WAN", context=context_text)
print(f"  Time: {time.time()-t0:.2f}s | Length: {len(explain_resp)}")
print(f"  Preview: {explain_resp[:300]}")
print()

--- ExplainHandler ---
  Time: 4.47s | Length: 3763
  Preview: Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Hôm nay chúng ta sẽ cùng nhau tìm hiểu về sự khác biệt giữa mạng LAN và mạng WAN nhé. Đây là hai khái niệm rất quan trọng trong lĩnh vực mạng máy tính mà chúng ta sẽ gặp nhiều trong chương trình Tin học THPT đấy.

### 1. Khái niệm cốt 



In [ ]:
# --- 6c. MCQHandler ---
print("--- MCQHandler ---")
mcq_h = MCQHandler()
t0 = time.time()
mcq_result = mcq_h.handle(query="Sinh 3 cau trac nghiem ve mang may tinh", context=context_text, num_questions=3)
gen_time = time.time() - t0
print(f"  Time: {gen_time:.2f}s")
if mcq_result:
    print(f"  Questions: {len(mcq_result.mcq)}")
    display = mcq_result.to_display_format()
    print(f"  Display:\n{display[:500]}")
else:
    print("  [!] Handler returned None")

--- MCQHandler ---
  Time: 2.66s
  Questions: 3
  Display:
Câu hỏi 1:
Trong lập trình, đại lượng lôgic thường được biểu diễn bằng những giá trị nào để ngắn gọn?

A. 1 và 0
B. Đúng và Sai
C. A và B
D. True và False

________________________________________

Câu hỏi 2:
Theo quy tắc xác định năm nhuận, năm nào sau đây KHÔNG phải là năm nhuận?

A. Năm 2000
B. Năm 1900
C. Năm 2024
D. Năm 3328

________________________________________

Câu hỏi 3:
Hành động nào sau đây thể hiện việc ứng xử nhân văn trên không gian mạng?

A. Lan truyền tin giả để câu view.
B. T


### 3.7 — Node: Question Validator

In [ ]:
from src.llm.validators.question_validator import QuestionValidator

if mcq_result:
    print("=" * 60)
    print("[NODE 7] Question Validator")
    print("=" * 60)
    
    validator = QuestionValidator()
    t0 = time.time()
    val_result = validator.validate(
        question_type="mcq",
        context=context_text,
        questions_json=json.dumps(mcq_result.model_dump())
    )
    val_time = time.time() - t0
    
    print(f"  Time: {val_time:.2f}s")
    print(f"  All valid: {val_result.all_valid}")
    print(f"  Approved: {len(val_result.approved_questions)}")
    print(f"  Validations: {len(val_result.validations)}")
    for v in val_result.validations:
        print(f"    - {v}")
else:
    print("[!] Skip validator -- mcq_result is None")

[NODE 7] Question Validator
  Time: 2.35s
  All valid: True
  Approved: 3
  Validations: 3
    - index=1 is_valid=True issues=[] fixed_question=None
    - index=2 is_valid=True issues=[] fixed_question=None
    - index=3 is_valid=True issues=[] fixed_question=None


---
## 4. Full Pipeline — Orchestrator.ask()

Chay toan bo pipeline end-to-end. Ket qua trace duoc ghi vao `pipeline_trace.log` dang **JSON**.

In [ ]:
from src.llm.memory import MemoryManager

# Reset memory truoc moi test
orch.memory = MemoryManager()

QUERY = "tổng hợp kiến thức tin học của lơp 12"

print("=" * 70)
print(f"FULL PIPELINE -- Query: '{QUERY}'")
print("=" * 70)

t0 = time.time()
response_chunks = []
for chunk in orch.ask(QUERY, ui_book="CD"):
    response_chunks.append(chunk)
total = time.time() - t0

full_response = "".join(response_chunks)
print(f"\n{'=' * 70}")
print(f"RESPONSE ({len(full_response)} chars, {total:.2f}s):")
print(f"{'=' * 70}")
print(full_response[:1000])
if len(full_response) > 1000:
    print(f"\n... [{len(full_response) - 1000} chars truncated]")

[21:59:37] INFO    | ============================================================
[21:59:37] INFO    | QUERY: 'tổng hợp kiến thức tin học của lơp 12'


FULL PIPELINE -- Query: 'tổng hợp kiến thức tin học của lơp 12'


[21:59:39] INFO    | IntentRouter: intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True
[21:59:39] INFO    | IntentRouter (1.45s): intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True
[21:59:39] INFO    | No current session, creating new
[21:59:39] INFO    | New session created: id=184abd0a, topic='kiến thức tin học lớp 12', intent=explain
[21:59:39] INFO    | Session: id=184abd0a, topic='kiến thức tin học lớp 12', msgs=0
[21:59:39] INFO    | ActionPlan: explain_concept (General concept explanation)
[21:59:39] INFO    | RAGAgent: strategy=broad | grade=12 | topic=kiến thức tin học lớp 12 | Query tổng quát: broad=True, grade_only=True, topic_broad=True
[21:59:39] INFO    | RAGAgent done: 30 chunks, 0.02s
[22:00:03] INFO    | Total time: 26.16s
[22:00:03] INFO    | ============================================================



RESPONSE (20597 chars, 26.16s):
Dang tim tai lieu de giai thich...

Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Thầy rất vui được đồng hành cùng em trong hành trình khám phá kiến thức Tin học lớp 12.

Em muốn thầy tổng hợp kiến thức Tin học lớp 12 đúng không nào? Lớp 12 là một năm học quan trọng với nhiều kiến thức mới và thú vị. Chúng ta sẽ cùng nhau đi qua các chủ đề chính nhé!

---

### **1. Trí tuệ nhân tạo (AI) và Học máy**

**1. Khái niệm cốt lõi:**

*   **Trí tuệ nhân tạo (AI):** Là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống có khả năng thực hiện các nhiệm vụ mà thông thường đòi hỏi trí tuệ con người, như học hỏi, giải quyết vấn đề, nhận dạng, và ra quyết định.
*   **Học máy (Machine Learning):** Là một nhánh của AI, tập trung vào việc phát triển các thuật toán cho phép máy tính "học" từ dữ liệu mà không cần được lập trình rõ ràng cho từng tác vụ.

**2. Giải thích chi tiết:**

*   **AI có tri thức, suy luận và học:**
    *   **Tri thứ

### 4.1 — Xem Pipeline Trace JSON

In [ ]:
print("=" * 70)
print("PIPELINE TRACE (pipeline_trace.log — JSON)")
print("=" * 70)
show_trace_log()

PIPELINE TRACE (pipeline_trace.log — JSON)
{
  "query": "tổng hợp kiến thức tin học của lơp 12",
  "timestamp": "2026-04-09 02:01:22",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "explain",
      "task_type": null,
      "topic": "kiến thức tin học lớp 12",
      "is_new_topic": true,
      "time_s": 0.82
    },
    {
      "node": "SessionManager",
      "session_id": "457f1671",
      "topic": "kiến thức tin học lớp 12",
      "intent": "explain",
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "explain_concept",
      "reason": "General concept explanation",
      "round_id": null
    },
    {
      "node": "RAG",
      "strategy": "broad",
      "chunks_returned": 30,
      "time_s": 0.01,
      "filter": {
        "grade": "12",
        "topic": "kiến thức tin học lớp 12"
      },
   

### 4.2 — Xem Debug Info (orchestrator.last_debug_info)

In [ ]:
print("=" * 70)
print("DEBUG INFO (orchestrator.last_debug_info)")
print("=" * 70)
show_debug_info(orch.last_debug_info)

DEBUG INFO (orchestrator.last_debug_info)
{
  "query": "tổng hợp kiến thức tin học của lơp 12",
  "timestamp": "2026-04-09 02:01:22",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "explain",
      "task_type": null,
      "topic": "kiến thức tin học lớp 12",
      "is_new_topic": true,
      "time_s": 0.82
    },
    {
      "node": "SessionManager",
      "session_id": "457f1671",
      "topic": "kiến thức tin học lớp 12",
      "intent": "explain",
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "explain_concept",
      "reason": "General concept explanation",
      "round_id": null
    },
    {
      "node": "RAG",
      "strategy": "broad",
      "chunks_returned": 30,
      "time_s": 0.01,
      "filter": {
        "grade": "12",
        "topic": "kiến thức tin học lớp 12"
      },
    

### 4.3 — Xem App Log (n dong cuoi)

In [ ]:
print("=" * 70)
print("APP LOG (last 20 lines)")
print("=" * 70)
show_app_log(20)

APP LOG (last 20 lines)
[2026-04-09 02:00:27] INFO    | chatbot.session_manager | No current session, creating new
[2026-04-09 02:00:27] INFO    | chatbot.session_manager | New session created: id=f76b2f3c, topic='', intent=chat
[2026-04-09 02:00:27] INFO    | chatbot.session_manager | Topic changed: '' -> 'Mang may tinh', creating new session
[2026-04-09 02:00:27] DEBUG   | chatbot.session_store | Session saved: f76b2f3c -> c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN\data\sessions\f76b2f3c.json
[2026-04-09 02:00:27] DEBUG   | chatbot.session_manager | Session archived: f76b2f3c
[2026-04-09 02:00:27] INFO    | chatbot.session_manager | New session created: id=5a540543, topic='Mang may tinh', intent=generate
[2026-04-09 02:00:27] DEBUG   | chatbot.session_manager | Keeping current session: 5a540543
[2026-04-09 02:01:22] INFO    | chatbot | ============================================================
[2026-04-09 02:01:22] INFO    | chatbot | QUERY: '

---
## 4.5 — Book Blocking Test

Kiem tra Orchestrator block khi query can tao RAG (generate/explain) nhung khong co book.

In [ ]:
orch.memory = MemoryManager()

QUERY = "Tao 3 cau hoi ve tri tue nhan tao"
print("=" * 70)
print(f"BOOK BLOCKING TEST -- Query: '{QUERY}'")
print("=" * 70)

block_resp = []
for chunk in orch.ask(QUERY, ui_book=None):
    block_resp.append(chunk)

print("\nRESPONSE:\n")
print("".join(block_resp))
print("\nDEBUG INFO:\n")
show_debug_info(orch.last_debug_info)


---
## 5. Multi-Query Test — Chuoi hoi thoai

Test pipeline lien tiep nhieu query de kiem tra session management.

In [ ]:
# Reset memory
orch.memory = MemoryManager()

queries = [
    "Xin chao ban la ai",
    "Giai thich mang may tinh cho toi",
    "Tao 3 cau trac nghiem ve chu de nay",
]

print("=" * 70)
print("MULTI-QUERY TEST")
print("=" * 70)

for i, q in enumerate(queries, 1):
    print(f"\n{'=' * 70}")
    print(f"[Query {i}/{len(queries)}]: {q}")
    print(f"{'=' * 70}")
    
    t0 = time.time()
    chunks = list(orch.ask(q, ui_book="KNTT"))
    total = time.time() - t0
    response = "".join(chunks)
    
    # Load JSON trace for this query
    trace_data = load_trace_json()
    steps = trace_data.get('steps', [])
    
    # Print summary per node
    for step in steps:
        node = step.get('node', '?')
        if node == 'ContextAnalyzer':
            print(f"  [1] ContextAnalyzer: enriched={step.get('enriched')}")
        elif node == 'IntentRouter':
            print(f"  [2] IntentRouter: intent={step.get('primary_intent')} | topic={step.get('topic')} | time={step.get('time_s')}s")
        elif node == 'SessionManager':
            print(f"  [3] SessionManager: id={step.get('session_id')} | topic={step.get('topic')} | msgs={step.get('total_messages')}")
        elif node == 'ActionPlanner':
            print(f"  [4] ActionPlanner: action={step.get('action')} | reason={step.get('reason')}")
        elif node == 'RAG':
            print(f"  [5] RAG: search={step.get('search_results')} | reranked={step.get('reranked')} | time={step.get('time_s')}s")
        elif node == 'Handler':
            print(f"  [6] Handler: action={step.get('action')} | status={step.get('status')} | {json.dumps({k:v for k,v in step.items() if k not in ('node','action','status')}, ensure_ascii=False)}")
    
    print(f"  Total: {total:.2f}s | Response: {len(response)} chars")
    print(f"  Preview: {response[:200]}")

[02:02:08] INFO    | ============================================================
[02:02:08] INFO    | QUERY: 'Xin chao ban la ai'


MULTI-QUERY TEST

[Query 1/3]: Xin chao ban la ai


[02:02:09] INFO    | IntentRouter: intent=chat, task_type=None, topic=None, is_new_topic=False
[02:02:09] INFO    | IntentRouter (0.99s): intent=chat, task_type=None, topic=None, is_new_topic=False
[02:02:09] INFO    | Session: id=457f1671, topic='kiến thức tin học lớp 12', msgs=2
[02:02:09] INFO    | ActionPlan: chat (Default chat intent)
[02:02:09] INFO    | RAGAgent: strategy=standard | grade=None | topic=None | Query cụ thể
[02:02:33] INFO    | RAGAgent done: 5 chunks, 23.91s
[02:02:34] INFO    | Total time: 26.02s
[02:02:34] INFO    | ============================================================
[02:02:34] INFO    | ============================================================
[02:02:34] INFO    | QUERY: 'Giai thich mang may tinh cho toi'


  [1] ContextAnalyzer: enriched=False
  [2] IntentRouter: intent=chat | topic=None | time=0.99s
  [3] SessionManager: id=457f1671 | topic=kiến thức tin học lớp 12 | msgs=2
  [4] ActionPlanner: action=chat | reason=Default chat intent
  [5] RAG: search=None | reranked=None | time=23.91s
  [6] Handler: action=chat | status=success | {"rag_chunks": 5, "chat_time_s": 1.1, "response_length": 156}
  Total: 26.02s | Response: 156 chars
  Preview: Chào bạn! 👋 Mình là EduBot, trợ lý học tập Tin học THPT Việt Nam. Mình ở đây để giúp bạn giải đáp các thắc mắc về môn Tin học dựa trên sách giáo khoa nhé! 😊

[Query 2/3]: Giai thich mang may tinh cho toi


[02:02:35] INFO    | IntentRouter: intent=explain, task_type=None, topic=mạng máy tính, is_new_topic=True
[02:02:35] INFO    | IntentRouter (0.90s): intent=explain, task_type=None, topic=mạng máy tính, is_new_topic=True
[02:02:35] INFO    | Topic changed: 'kiến thức tin học lớp 12' -> 'mạng máy tính', creating new session
[02:02:35] INFO    | New session created: id=66342982, topic='mạng máy tính', intent=explain
[02:02:35] INFO    | Session: id=66342982, topic='mạng máy tính', msgs=0
[02:02:35] INFO    | ActionPlan: explain_concept (General concept explanation)
[02:02:35] INFO    | RAGAgent: strategy=broad | grade=None | topic=mạng máy tính | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[02:02:35] INFO    | RAGAgent done: 16 chunks, 0.00s
[02:02:41] INFO    | Total time: 7.53s
[02:02:41] INFO    | ============================================================
[02:02:41] INFO    | ============================================================
[02:02:41] INFO    | QUERY:

  [1] ContextAnalyzer: enriched=False
  [2] IntentRouter: intent=explain | topic=mạng máy tính | time=0.9s
  [3] SessionManager: id=66342982 | topic=mạng máy tính | msgs=0
  [4] ActionPlanner: action=explain_concept | reason=General concept explanation
  [5] RAG: search=None | reranked=None | time=0.0s
  [6] Handler: action=explain_concept | status=success | {"rag_chunks": 16, "explain_time_s": 6.6, "response_length": 6213}
  Total: 7.54s | Response: 6249 chars
  Preview: Dang tim tai lieu de giai thich...

Chào em, thầy rất vui được giải thích về mạng máy tính cho em đây! Mạng máy tính là một khái niệm rất quan trọng trong Tin học THPT, nó giúp chúng ta hiểu rõ hơn về

[Query 3/3]: Tao 3 cau trac nghiem ve chu de nay


[02:02:42] INFO    | IntentRouter: intent=generate, task_type=mcq, topic=None, is_new_topic=True
[02:02:42] INFO    | IntentRouter (0.88s): intent=generate, task_type=mcq, topic=None, is_new_topic=True
[02:02:42] INFO    | Session: id=66342982, topic='mạng máy tính', msgs=2
[02:02:42] INFO    | ActionPlan: generate_quiz (task_type=mcq)
[02:02:42] INFO    | RAGAgent: strategy=standard | grade=None | topic=None | Query cụ thể
[02:03:27] INFO    | RAGAgent done: 5 chunks, 44.80s
[02:03:27] INFO    | RAG Search: 5 chunks (44.80s)
[02:03:27] INFO    | Generate: type=mcq, num=3
[02:03:30] INFO    | Handler.handle() -> 2.47s (attempt 1)
[02:03:32] INFO    | Validator: all_valid=True, approved=3 (2.47s)
[02:03:32] INFO    | Saved 3 questions to round 0 (total rounds: 1)
[02:03:32] INFO    | Total time: 50.66s
[02:03:32] INFO    | ============================================================


  [1] ContextAnalyzer: enriched=False
  [2] IntentRouter: intent=generate | topic=None | time=0.88s
  [3] SessionManager: id=66342982 | topic=mạng máy tính | msgs=2
  [4] ActionPlanner: action=generate_quiz | reason=task_type=mcq
  [5] RAG: search=None | reranked=None | time=44.8s
  [6] Handler: action=generate_quiz | status=success | {"handler": "MCQHandler", "task_type": "mcq", "num_questions": 3, "rag_chunks": 5, "rag_time_s": 44.8, "context_length": 3290, "generation_time_s": 2.47, "generation_attempts": 1, "validator_all_valid": true, "validator_approved": 3, "validator_time_s": 2.47, "questions_saved": 3, "round_id": 0}
  Total: 50.66s | Response: 958 chars
  Preview: Dang tim kiem tai lieu lien quan...Dang soan 3 cau hoi MCQ...Dang kiem duyet chat luong...

Câu hỏi 1:
Trong Microsoft Access, để đóng một cơ sở dữ liệu, em cần thực hiện hành động nào?

A. Chọn nút l


### 5.1 — Xem Trace JSON cua query cuoi

In [ ]:
show_trace_log()

{
  "query": "Tao 3 cau trac nghiem ve chu de nay",
  "timestamp": "2026-04-09 02:02:41",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "generate",
      "task_type": "mcq",
      "topic": null,
      "is_new_topic": true,
      "time_s": 0.88
    },
    {
      "node": "SessionManager",
      "session_id": "66342982",
      "topic": "mạng máy tính",
      "intent": "explain",
      "total_messages": 2,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "generate_quiz",
      "reason": "task_type=mcq",
      "round_id": null
    },
    {
      "node": "RAG",
      "strategy": "standard",
      "chunks_returned": 5,
      "time_s": 44.8,
      "filter": {
        "grade": null,
        "topic": null
      },
      "reason": "Query cụ thể"
    },
    {
      "node": "Handler",
      "handler": "MCQHandler",
      "action":

---
## 6. Custom Query — Tu nhap de test

In [ ]:
# === THAY DOI QUERY TAI DAY ===
CUSTOM_QUERY = "Tao 3 cau dung sai ve an toan thong tin"
# ===============================

print(f"Query: '{CUSTOM_QUERY}'")
print("=" * 70)

t0 = time.time()
chunks = list(orch.ask(CUSTOM_QUERY))
total = time.time() - t0
response = "".join(chunks)

print(f"\nResponse ({len(response)} chars, {total:.2f}s):")
print("=" * 70)
print(response[:1500])

print("\n" + "=" * 70)
print("PIPELINE TRACE JSON:")
print("=" * 70)
show_trace_log()

[02:03:32] INFO    | ============================================================
[02:03:32] INFO    | QUERY: 'Tao 3 cau dung sai ve an toan thong tin'


Query: 'Tao 3 cau dung sai ve an toan thong tin'


[02:03:33] INFO    | IntentRouter: intent=generate, task_type=true_false, topic=an toan thong tin, is_new_topic=True
[02:03:33] INFO    | IntentRouter (0.93s): intent=generate, task_type=true_false, topic=an toan thong tin, is_new_topic=True
[02:03:33] INFO    | Topic changed: 'mạng máy tính' -> 'an toan thong tin', creating new session
[02:03:33] INFO    | New session created: id=bbfd614a, topic='an toan thong tin', intent=generate
[02:03:33] INFO    | Session: id=bbfd614a, topic='an toan thong tin', msgs=0
[02:03:33] INFO    | ActionPlan: generate_quiz (task_type=true_false)
[02:03:33] INFO    | RAGAgent: strategy=standard | grade=None | topic=an toan thong tin | Query cụ thể
[02:04:19] INFO    | RAGAgent done: 5 chunks, 45.85s
[02:04:19] INFO    | RAG Search: 5 chunks (45.86s)
[02:04:19] INFO    | Generate: type=true_false, num=3
[02:04:21] INFO    | Handler.handle() -> 2.11s (attempt 1)
[02:04:23] INFO    | Validator: all_valid=True, approved=3 (2.10s)
[02:04:23] INFO    | Saved 3 


Response (628 chars, 51.02s):
Dang tim kiem tai lieu lien quan...Dang soan 3 cau hoi TRUE_FALSE...Dang kiem duyet chat luong...

❓ Câu 1:
   Switch sử dụng địa chỉ IP để chuyển tiếp các gói dữ liệu giữa các thiết bị trong mạng cục bộ.
   → Đúng hay Sai?

________________________________________

❓ Câu 2:
   Việc chia sẻ thông tin bịa đặt về dịch bệnh trên mạng xã hội có thể vi phạm pháp luật.
   → Đúng hay Sai?

________________________________________

❓ Câu 3:
   Bảng định tuyến (routing table) trong Router chứa địa chỉ MAC của các thiết bị trong mạng cục bộ.
   → Đúng hay Sai?

________________________________________


[Round 1] Da tao 3 cau hoi.

PIPELINE TRACE JSON:
{
  "query": "Tao 3 cau dung sai ve an toan thong tin",
  "timestamp": "2026-04-09 02:03:32",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "generate",
      "task_type": "true_false",
      "topic": "an toan thong tin",


---
## 7. Timing Benchmark — Do thoi gian tung node

In [ ]:
# Reset memory
orch.memory = MemoryManager()

benchmark_query = "Tao 3 cau trac nghiem ve he dieu hanh"

print(f"Benchmark: '{benchmark_query}'")
print("=" * 70)

chunks = list(orch.ask(benchmark_query))

# Load JSON trace
trace_data = load_trace_json()
steps = trace_data.get('steps', [])

print(f"\n{'Node':<25} {'Time (s)':>10}")
print('=' * 37)

for step in steps:
    node = step.get('node', '?')
    # Find timing field
    t = step.get('time_s') or step.get('generation_time_s') or step.get('chat_time_s') or step.get('explain_time_s') or step.get('slide_time_s') or step.get('scorer_time_s')
    label = node
    if node == 'Handler':
        label = f"Handler:{step.get('action','?')}"
    if t is not None:
        bar = '#' * int(t * 2)
        print(f"  {label:<23} {t:>8.2f}s  {bar}")
    else:
        print(f"  {label:<23}      -")

total = trace_data.get('total_time_s', 0)
print('=' * 37)
print(f"  {'TOTAL':<23} {total:>8.2f}s")

# Response info
resp = trace_data.get('response', {})
print(f"\nResponse: {resp.get('length', 0)} chars")

[02:04:23] INFO    | ============================================================
[02:04:23] INFO    | QUERY: 'Tao 3 cau trac nghiem ve he dieu hanh'


Benchmark: 'Tao 3 cau trac nghiem ve he dieu hanh'


[02:04:24] INFO    | IntentRouter: intent=generate, task_type=mcq, topic=hệ điều hành, is_new_topic=True
[02:04:24] INFO    | IntentRouter (0.97s): intent=generate, task_type=mcq, topic=hệ điều hành, is_new_topic=True
[02:04:24] INFO    | Topic changed: 'an toan thong tin' -> 'hệ điều hành', creating new session
[02:04:24] INFO    | New session created: id=5e35c39a, topic='hệ điều hành', intent=generate
[02:04:24] INFO    | Session: id=5e35c39a, topic='hệ điều hành', msgs=0
[02:04:24] INFO    | ActionPlan: generate_quiz (task_type=mcq)
[02:04:24] INFO    | RAGAgent: strategy=standard | grade=None | topic=hệ điều hành | Query cụ thể
[02:04:53] INFO    | RAGAgent done: 5 chunks, 29.14s
[02:04:53] INFO    | RAG Search: 5 chunks (29.15s)
[02:04:53] INFO    | Generate: type=mcq, num=3
[02:04:56] INFO    | Handler.handle() -> 2.43s (attempt 1)
[02:04:58] INFO    | Validator: all_valid=True, approved=3 (2.34s)
[02:04:58] INFO    | Saved 3 questions to round 0 (total rounds: 1)
[02:04:58] INFO


Node                        Time (s)
  ContextAnalyzer              -
  IntentRouter                0.97s  #
  SessionManager               -
  ActionPlanner                -
  RAG                        29.14s  ##########################################################
  Handler:generate_quiz       2.43s  ####
  TOTAL                      34.91s

Response: 886 chars
